In [1]:
import os

# Ensure the static folder exists
os.makedirs("static", exist_ok=True)

# OBJ data for a simple 3D pyramid (placeholder for a plant)
obj_data = """
# Dummy Low-Poly Plant
v 0.0 2.0 0.0
v -1.0 0.0 1.0
v 1.0 0.0 1.0
v 1.0 0.0 -1.0
v -1.0 0.0 -1.0
f 1 2 3
f 1 3 4
f 1 4 5
f 1 5 2
f 5 4 3 2
"""

with open("static/plant_model.obj", "w") as f:
    f.write(obj_data.strip())

print("✅ Saved dummy 3D model to static/plant_model.obj")

✅ Saved dummy 3D model to static/plant_model.obj


In [2]:
import numpy as np
from scipy.io import wavfile
from pathlib import Path
import random

# Point this to your dataset directory
PROCESSED2_DIR = Path(r"G:\AgriTech\agri_projects\plant_pulse_3d\PlantPulse_3D\data\processed2")
SAMPLE_RATE = 500000  # 500 kHz

def generate_test_wav(output_filename, counts_dict):
    """
    Pulls specific counts of audio events, shuffles them, and merges them into one .wav
    """
    all_selected_paths = []
    
    # Gather the requested number of files from each category
    for folder, count in counts_dict.items():
        if count == 0:
            continue
            
        folder_path = PROCESSED2_DIR / folder
        if not folder_path.exists():
            print(f"Warning: Folder {folder} not found!")
            continue
            
        npy_files = list(folder_path.glob("*.npy"))
        if not npy_files:
            print(f"No files found in {folder}")
            continue
            
        # Allow repetition if we ask for more files than exist in the folder
        if count > len(npy_files):
            selected = random.choices(npy_files, k=count)
        else:
            selected = random.sample(npy_files, count)
            
        all_selected_paths.extend(selected)

    if not all_selected_paths:
        print(f"Error: No data found to merge for {output_filename}.")
        return

    # Shuffle the events so the AI sees them in a random, natural order
    random.shuffle(all_selected_paths)
    
    combined_signals = []
    for file_path in all_selected_paths:
        signal = np.load(file_path).astype(np.float32)
        combined_signals.append(signal)
        # Add a tiny 500-sample gap of silence between events
        silence = np.zeros(500, dtype=np.float32)
        combined_signals.append(silence)

    # Concatenate into one continuous audio stream
    master_signal = np.concatenate(combined_signals)
    
    # Normalize to prevent clipping distortion
    max_val = np.max(np.abs(master_signal))
    if max_val > 0:
        master_signal = master_signal / max_val

    # Convert to 16-bit PCM for standard WAV format
    scaled_signal = (master_signal * 32767).astype(np.int16)
    
    output_path = Path(output_filename)
    wavfile.write(output_path, SAMPLE_RATE, scaled_signal)
    
    print(f"✅ Saved: {output_filename}")
    print(f"   Recipe: {counts_dict}")
    print(f"   Duration: {len(master_signal) / SAMPLE_RATE:.4f} seconds\n")

if __name__ == "__main__":
    print("🧪 Generating PlantPulse Testing Suite...\n")
    
    # CASE 1: Pure Silence/Background (Should trigger "Normal (Background)")
    generate_test_wav("test_1_pure_normal.wav", {
        "empty_pot": 15, "tomato_cut": 0, "tomato_dry": 0
    })
    
    # CASE 2: Pure Mechanical Damage (Should trigger "Stress (Cut)")
    generate_test_wav("test_2_pure_cut.wav", {
        "empty_pot": 5, "tomato_cut": 8, "tomato_dry": 0
    })
    
    # CASE 3: Pure Hydric Stress (Should trigger "Stress (Dehydration)")
    generate_test_wav("test_3_pure_dry.wav", {
        "empty_pot": 4, "tomato_cut": 0, "tomato_dry": 9
    })
    
    # CASE 4: Mixed Stress (Ratio = 3/4 = 75% -> Should trigger "Mixed Stress")
    generate_test_wav("test_4_true_mixed.wav", {
        "empty_pot": 2, "tomato_cut": 4, "tomato_dry": 3
    })
    
    # CASE 5: Anomaly Filter Test (Ratio = 1/10 = 10% -> Should ignore the 1 dry and trigger "Stress (Cut)")
    generate_test_wav("test_5_anomaly_filter.wav", {
        "empty_pot": 6, "tomato_cut": 10, "tomato_dry": 1
    })
    
    print("🎉 All test files generated successfully in the root directory!")

🧪 Generating PlantPulse Testing Suite...

✅ Saved: test_1_pure_normal.wav
   Recipe: {'empty_pot': 15, 'tomato_cut': 0, 'tomato_dry': 0}
   Duration: 0.0450 seconds

✅ Saved: test_2_pure_cut.wav
   Recipe: {'empty_pot': 5, 'tomato_cut': 8, 'tomato_dry': 0}
   Duration: 0.0390 seconds

✅ Saved: test_3_pure_dry.wav
   Recipe: {'empty_pot': 4, 'tomato_cut': 0, 'tomato_dry': 9}
   Duration: 0.0390 seconds

✅ Saved: test_4_true_mixed.wav
   Recipe: {'empty_pot': 2, 'tomato_cut': 4, 'tomato_dry': 3}
   Duration: 0.0270 seconds

✅ Saved: test_5_anomaly_filter.wav
   Recipe: {'empty_pot': 6, 'tomato_cut': 10, 'tomato_dry': 1}
   Duration: 0.0510 seconds

🎉 All test files generated successfully in the root directory!


In [5]:
import trimesh

import trimesh

# Pass the exact absolute path to your static folder
file_path = r"G:\AgriTech\agri_projects\plant_pulse_3d\PlantPulse_3D\static\crops_low_poly.glb"

scene_or_mesh = trimesh.load(file_path)

# Inspect the structure
if isinstance(scene_or_mesh, trimesh.Scene):
    print("Scene nodes found!")
    for name, geom in scene_or_mesh.geometry.items():
        print(f"Geometry Name: {name}, Vertices: {len(geom.vertices)}")
else:
    print("Loaded as a single mesh.")

# 2. Inspect the structure (check what sub-meshes or nodes exist)
if isinstance(scene_or_mesh, trimesh.Scene):
    print("Scene nodes:", scene_or_mesh.graph.nodes)
    
    # Example: If the unwanted crops are separate geometries, 
    # you can iterate through and drop them by name or index.
    # Let's see what geometry keys we have:
    for name, geom in scene_or_mesh.geometry.items():
        print(f"Geometry Name: {name}, Vertices: {len(geom.vertices)}")
        
        # If a geometry name contains 'wheat', 'carrot', or 'potato', 
        # you can remove it from the geometry dictionary:
        # if any(crop in name.lower() for crop in ['wheat', 'carrot', 'potato']):
        #     del scene_or_mesh.geometry[name]
else:
    print("Loaded as a single mesh.")

# 3. Export the cleaned environment containing only the dirt land
# scene_or_mesh.export('clean_farm_land.glb')
print("Ready to filter and prepare your multi-plant layout coordinates!")

Scene nodes found!
Geometry Name: Carrot_F1_Carrot_0, Vertices: 128
Geometry Name: Carrot_F2_Carrot_0, Vertices: 128
Geometry Name: Carrot_F3_Carrot_0, Vertices: 1360
Geometry Name: Potatoe_F1_Potatoe_0, Vertices: 128
Geometry Name: Potatoe_F2_Potatoe_0, Vertices: 128
Geometry Name: Potatoe_F3_Potatoe_0, Vertices: 1884
Geometry Name: Tomatoe_F1_Tomatoe_0, Vertices: 6528
Geometry Name: Tomatoe_F2_Tomatoe_0, Vertices: 30464
Geometry Name: Tomatoe_F3_Tomatoe_0, Vertices: 30464
Geometry Name: Wheat_F1_Wheat_0, Vertices: 7104
Geometry Name: Wheat_F2_Wheat_0, Vertices: 65534
Geometry Name: Wheat_F2_Wheat_1, Vertices: 65532
Geometry Name: Wheat_F2_Wheat_0_2, Vertices: 65532
Geometry Name: Wheat_F2_Wheat_0_3, Vertices: 7422
Geometry Name: Wheat_F3_Wheat_0, Vertices: 65532
Geometry Name: Wheat_F3_Wheat_1, Vertices: 65533
Geometry Name: Wheat_F3_Wheat_0_2, Vertices: 65532
Geometry Name: Wheat_F3_Wheat_0_3, Vertices: 7551
Geometry Name: Soil_Dirt_0, Vertices: 192
Geometry Name: Soil.001_Dirt_0, V

In [6]:
# Create a copy of the scene graph or geometry keys to modify
keys_to_remove = []
for name, geom in scene_or_mesh.geometry.items():
    # If it's NOT soil, mark it for deletion
    if not "Soil" in name:
        keys_to_remove.append(name)

# Remove unwanted geometries
for key in keys_to_remove:
    del scene_or_mesh.geometry[key]

# Export the clean, soil-only land
output_path = r"G:\AgriTech\agri_projects\plant_pulse_3d\PlantPulse_3D\static\clean_soil_land.glb"
scene_or_mesh.export(output_path)
print("Success! Clean soil-only land exported to static folder.")

Success! Clean soil-only land exported to static folder.


In [3]:
import numpy as np
from scipy.io import wavfile
from pathlib import Path
import random

# --- 1. CONFIGURATION PATHS ---
RAW_DIR = Path(r"G:\AgriTech\agri_projects\plant_pulse_3d\PlantPulse_3D\data\raw2\PlantSounds")
OUT_DIR = Path(r"G:\AgriTech\agri_projects\plant_pulse_3d\PlantPulse_3D\data\test_audios")
OUT_DIR.mkdir(parents=True, exist_ok=True)

empty_files = list((RAW_DIR / "Empty Pot").glob("*.wav"))
cut_files = list((RAW_DIR / "Tomato Cut").glob("*.wav"))
dry_files = list((RAW_DIR / "Tomato Dry").glob("*.wav"))

def read_wav(filepath):
    sr, data = wavfile.read(filepath)
    if len(data.shape) > 1:
        data = data[:, 0]
    return sr, data

def generate_test_track(event_files, output_name, num_events=2, track_length_seconds=0.2):
    sr = 500000 
    total_samples = int(sr * track_length_seconds) # 100,000 samples
    
    # 1. Fill the track with pure Empty Pot background (RAW, no fades)
    track = np.zeros(total_samples, dtype=np.float32)
    idx = 0
    while idx < total_samples:
        _, bg_data = read_wav(random.choice(empty_files))
        end = min(idx + len(bg_data), total_samples)
        track[idx:end] = bg_data[:end - idx]
        idx += len(bg_data)
        
    # 2. Inject the stress pops EXACTLY aligned to the AI's windowing
    if event_files and num_events > 0:
        available_windows = total_samples // 1000
        
        # Pick random windows to inject the pops
        injection_windows = random.sample(range(1, available_windows - 1), num_events)
        
        for win_idx in injection_windows:
            _, event_data = read_wav(random.choice(event_files))
            
            # Force the event to be exactly 1000 samples (just like your training loader)
            if len(event_data) > 1000:
                event_data = event_data[:1000]
            elif len(event_data) < 1000:
                event_data = np.pad(event_data, (0, 1000 - len(event_data)), mode='constant')
            
            # Snap to the exact grid the AI uses
            start_idx = win_idx * 1000
            end_idx = start_idx + 1000
            
            # OVERWRITE the background to preserve the exact frequency signature
            track[start_idx:end_idx] = event_data

    # 3. Save as standard 16-bit WAV
    out_path = OUT_DIR / output_name
    wavfile.write(out_path, sr, track.astype(np.int16))
    print(f"✅ Generated: {out_path.name} | Duration: {track_length_seconds}s | Pops Injected: {num_events}")

# --- 3. GENERATE THE 4 FAST TEST FILES ---
print("Initializing EXACT Match Synthetic Audio Generator...\n")

# 1. Normal (0.2 seconds background noise, 0 pops)
generate_test_track([], "01_test_normal.wav", num_events=0)

# 2. Cut Stress (0.2 seconds, 3 severe stem cut pops)
generate_test_track(cut_files, "02_test_cut.wav", num_events=3)

# 3. Dry Stress (0.2 seconds, 4 dehydration pops)
generate_test_track(dry_files, "03_test_dry.wav", num_events=4)

# 4. Mixed Stress (0.2 seconds, 6 overlapping pops)
mixed_files = cut_files + dry_files
generate_test_track(mixed_files, "04_test_mixed.wav", num_events=6)

print(f"\nFiles fixed and saved! Ready for fast, accurate inference.")

Initializing EXACT Match Synthetic Audio Generator...

✅ Generated: 01_test_normal.wav | Duration: 0.2s | Pops Injected: 0
✅ Generated: 02_test_cut.wav | Duration: 0.2s | Pops Injected: 3
✅ Generated: 03_test_dry.wav | Duration: 0.2s | Pops Injected: 4
✅ Generated: 04_test_mixed.wav | Duration: 0.2s | Pops Injected: 6

Files fixed and saved! Ready for fast, accurate inference.
